In [1]:
import chromadb
client = chromadb.Client()
from chromadb.utils import embedding_functions

In [3]:
# embedding_functions??
# embedding_functions.get_builtins()
# embedding_functions.SentenceTransformerEmbeddingFunction??

In [4]:
from myimports import *
import utils as ut

# 3. Load datasets
train_dataset = load_dataset("json", data_files="../data/trn_with_hard_negatives.json", split="train")
eval_dataset = load_dataset("json", data_files="../data/eval_with_hard_negatives.json", split="train")
test_dataset = load_dataset("json", data_files="../data/tst_with_hard_negatives.json", split="train")

# generate data for informationretreival evaluator
corpus_dataset,corpus_mapper=ut.get_corpus_and_corpus_mapper(train_dataset, eval_dataset, test_dataset, dup_col='positive')

#collect all positives from train,eval,test
corpus = dict(
    zip(corpus_dataset["id"], corpus_dataset["positive"])
)  # Our corpus (cid => document)


/home/kperkins411/anaconda3/envs/p311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /home/kperkins411/.cache/huggingface/token
Login successful
len(corpus_dataset) before dropping duplicates:102812
dropped 99644 duplicate rows. have 3168 rows left
len(corpus_dataset) after dropping duplicates:3168


In [5]:
#get uploaded fine tuned embedder
st_ef=embedding_functions.SentenceTransformerEmbeddingFunction("kperkins411/msmarco-distilbert-base-v2_triplet_legal",device='cpu')

#or from a local model
# st_ef=embedding_functions.SentenceTransformerEmbeddingFunction("./models/msmarco-distilbert-base-v2_triplet/final",device='cpu')

# Create a new chroma collection
st_collection = client.get_or_create_collection(name="st_embeddings", embedding_function=st_ef)

#add all corpus values to collection
st_collection.add(
    documents=list(corpus.values()),
    ids=[str(id) for id in list(corpus.keys())])

/home/kperkins411/anaconda3/envs/p311/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [6]:
#A single query
results = st_collection.query(
    # query_texts=test_dataset[0]['anchor'], #query single text
    query_texts=test_dataset['anchor'],  # Query all texts
    n_results=20
)
# results

In [8]:
#Lets see how it performs on multiple queries
from sentence_transformers import CrossEncoder
from tqdm import tqdm
import numpy as np
from collections import defaultdict

class track_stats:
    """
    A class to track statistics for document ranking.

    Attributes:
    - test_dataset (dict): The test dataset containing positive examples.
    - corpus_mapper (dict): A mapper to map document indices to their corresponding documents.
    - stats (defaultdict): A dictionary to store the statistics.

    Methods:
    - __init__(self, test_dataset, corpus_mapper): Initializes the track_stats object.
    - __call__(self, i, max_score): Updates the statistics based on the given index and maximum score.
    - print_stats(self): Prints the statistics.
    """

    def __init__(self, test_dataset, corpus_mapper):
        self.test_dataset = test_dataset
        self.corpus_mapper = corpus_mapper
        self.stats = defaultdict(int) #defaults to 0

    def __call__(self, i, max_score):
        """
        Updates the statistics based on the given index and maximum score.

        Parameters:
        - i (int): The index of the example.
        - max_score (int): The maximum score obtained for the example.
        """
        self.stats['totals'] += 1
        if max_score != 0:
            correct_doc = self.corpus_mapper[self.test_dataset['positive'][i]]
            original_choice = self.corpus_mapper[results['documents'][i][0]]
            reranked_choice = self.corpus_mapper[results['documents'][i][max_score]]
            if correct_doc == reranked_choice:
                self.stats['correctly_reranked'] += 1
            else:
                if correct_doc == original_choice:
                    self.stats['reranked_incorrectly'] += 1
                else:
                    self.stats['both_incorrect'] += 1

    def print_stats(self):
        """
        Prints the statistics.
        """
        print(f"Correct original predictions {self.stats['totals'] - self.stats['correctly_reranked'] - self.stats['reranked_incorrectly'] - self.stats['both_incorrect']} out of {self.stats['totals']}")
        print(f"correctly reranked {self.stats['correctly_reranked']} out of {self.stats['totals']}")
        print(f"incorrectly reranked {self.stats['reranked_incorrectly']} out of {self.stats['totals']}")
        print(f"both_incorrect {self.stats['both_incorrect']} out of {self.stats['totals']}")

#stat tracker    
ts = track_stats(test_dataset,corpus_mapper)

#this is not fine tuned!
model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', max_length=512)

# rerank the results with original query and documents returned from Chroma
for i in tqdm(range(len(test_dataset))):
    scores = model.predict([(test_dataset['anchor'][i], doc) for doc in results["documents"][i]])
    max_score=np.argmax(scores)
    ts(i,max_score)
ts.print_stats()

/home/kperkins411/anaconda3/envs/p311/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
  0%|          | 0/8235 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
 

In [ ]:
# corpus_mapper[results['documents'][1][np.argmax(scores)]]
i=15
print(f"reranker change for row {i}, correct doc={corpus_mapper[test_dataset['positive'][i]]}, reranked choice={corpus_mapper[results['documents'][i][np.argmax(scores)]]}, original choice={corpus_mapper[test_dataset['positive'][i]]}")


reranker change for row 15, correct doc=21675, reranked choice=21675, original choice=21675


In [ ]:
# #from https://medium.com/@rossashman/the-art-of-rag-part-3-reranking-with-cross-encoders-688a16b64669
# def reranker(query, hits):
#     from sentence_transformers import CrossEncoder
    
#     # To refine the results, we use a CrossEncoder. A CrossEncoder gets both inputs (input_question, retrieved_question)
#     # and outputs a score 0...1 indicating the similarity.
#     cross_encoder_model = CrossEncoder("cross-encoder/stsb-roberta-base")

#     # Now, do the re-ranking with the cross-encoder
#     sentence_pairs = [[query, hit["text"]] for hit in hits]
#     similarity_scores = cross_encoder_model.predict(sentence_pairs)
    
#     for idx in range(len(hits)):
#         hits[idx]["cross-encoder_score"] = similarity_scores[idx]

#     # Sort list by CrossEncoder scores
#     hits = sorted(hits, key=lambda x: x["cross-encoder_score"], reverse=True)
#     print("Top 5 hits with CrossEncoder:")
#     for hit in hits:
#         print("\t{:.3f}\t{}".format(hit["cross-encoder_score"], hit["_id"]))

#     print("\n\n========\n")


# #from https://medium.com/@rossashman/the-art-of-rag-with-atlas-part-2-hybrid-retrieval-77631457b565
# def weighted_reciprocal_rank(doc_lists):
#         """
#         This is a modified version of the fuction in the langchain repo
#         https://github.com/langchain-ai/langchain/blob/master/libs/langchain/langchain/retrievers/ensemble.py
        
#         Perform weighted Reciprocal Rank Fusion on multiple rank lists.
#         You can find more details about RRF here:
#         https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf

#         Args:
#             doc_lists: A list of rank lists, where each rank list contains unique items.

#         Returns:
#             list: The final aggregated list of items sorted by their weighted RRF
#                     scores in descending order.
#         """
#         c=60 #c comes from the paper
#         weights=[1]*len(doc_lists) #you can apply weights if you like, here they are all the same, ie 1
        
#         if len(doc_lists) != len(weights):
#             raise ValueError(
#                 "Number of rank lists must be equal to the number of weights."
#             )

#         # Create a union of all unique documents in the input doc_lists
#         all_documents = set()
#         for doc_list in doc_lists:
#             for doc in doc_list:
#                 all_documents.add(doc["text"])

#         # Initialize the RRF score dictionary for each document
#         rrf_score_dic = {doc: 0.0 for doc in all_documents}

#         # Calculate RRF scores for each document
#         for doc_list, weight in zip(doc_lists, weights):
#             for rank, doc in enumerate(doc_list, start=1):
#                 rrf_score = weight * (1 / (rank + c))
#                 rrf_score_dic[doc["text"]] += rrf_score

#         # Sort documents by their RRF scores in descending order
#         sorted_documents = sorted(
#             rrf_score_dic.keys(), key=lambda x: rrf_score_dic[x], reverse=True
#         )

#         # Map the sorted page_content back to the original document objects
#         page_content_to_doc_map = {
#             doc["text"]: doc for doc_list in doc_lists for doc in doc_list
#         }
#         sorted_docs = [
#             page_content_to_doc_map[page_content] for page_content in sorted_documents
#         ]

#         return sorted_docs